In [1]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks
pd.options.plotting.backend = 'plotly'  # setting plotly as the backend for pandas plotting

# Add parent directory to sys.path so local module 'synthetic' can be imported
import sys
from pathlib import Path # for path manipulations
# Move three levels up from current working directory to reach workspace root
workspace_root = Path.cwd().parent.parent.parent.resolve() 
smx_dir = workspace_root / 'smx'  # Path to smx folder
if str(smx_dir) not in sys.path: # check to avoid duplicates
    sys.path.insert(0, str(smx_dir)) # insert at the start of sys.path to prioritize local modules

# Loading a soil spectral dataset based on X-ray fluorescence (XRF)
data_complete = pd.read_csv(f'{workspace_root}/real_datasets/xrf/soil.csv', sep=';') # local copy of Toledo 2022 dataset (os ... indica para omitir o caminho longo)
data = data_complete.loc[:, '1.32':'13.1']

# Split dataset by class and create calibration/prediction sets using Kennard-Stone (as in original pipeline)
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.loc[:, '1.32':'13.1'], test_size=0.30)  # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.loc[:, '1.32':'13.1'], test_size=0.30)  # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True)  # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0])  # target for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0])  # target for prediction set

# preprocessings
import preprocessings as prepr  # preprocessing methods for XRF data

Xcalclass_prep, mean_calclass, mean_calclass_poisson  = prepr.poisson(Xcalclass, mc=True)
Xpredclass_prep = ((Xpredclass/np.sqrt(mean_calclass)) - mean_calclass_poisson)

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-02-26 15:03:15,359 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-02-26 15:03:15,374 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur

In [2]:
from modeling import svm_optimized

svm_model = svm_optimized(Xcalclass_prep, ycalclass, Xpredclass_prep, ypredclass, aim='classification', kernel='rbf')
svm_model[0]

,Model,Accuracy Cal,Sensitivity Cal,Specificity Cal,CM Cal,Accuracy Pred,Sensitivity Pred,Specificity Pred,CM Pred
0,SVC,0.925676,0.943662,0.909091,"[[70, 7], [4, 67]]",0.875,0.967742,0.787879,"[[26, 7], [1, 30]]"


## Definição das Zonas Espectrais

In [3]:
# establishing spectral cuts based on expert knowledge of XRF spectra
spectral_cuts = [
('Al', 1.33, 1.63),
('Si', 1.63, 1.86),
('P', 1.86, 2.19),
('S', 2.19, 2.55),
('Rh L + Ar', 2.55, 3.21),
('K', 3.21, 3.53),
('Ca ka', 3.53, 3.84),
('Ca kb', 3.84, 4.37),
('Ti ka', 4.37, 4.75),
('Ti kb', 4.75, 5.12),
('Cr', 5.12, 5.77),
('Mn', 5.77, 6.13),
('Fe ka', 6.13, 6.80),
('Fe kb', 6.80, 7.30),
('background1', 7.30, 7.91),
('Cu', 7.91, 8.20),
('background2', 8.20, 10.69),
('Fe ka + Ti ka', 10.69, 11.14),
('background3', 11.14, 12.55),
('sum Fe' , 12.55, 13.1)
]

## Pvector and SHAP

In [4]:
X_sv = svm_model[3].support_vectors_            # shape (n_SV, n_features)
alpha_dual = svm_model[3].dual_coef_.ravel()    # shape (n_SV,)

# Calculando os coeficientes p usando os vetores de suporte e os multiplicadores de Lagrange
pvetor = pd.DataFrame({'energia' : Xcalclass.columns,
                       'importance': (X_sv.T) @ alpha_dual})
pvetor['importance'] = np.abs(pvetor['importance'])
pvector_df = pd.DataFrame({
    'energy': pvetor['energia'],
    'Pvector': pvetor['importance'].values
})
pvector_df = pvector_df.sort_values(by='Pvector', ascending=False).reset_index(drop=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in pvector_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
pvector_df['Zone'] = pvector_df['energy'].map(energy_to_zone_vip)
pvector_unique_df = pvector_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
pvector_unique_df

,energy,Pvector,Zone
0,3.7,23.449350,Ca ka
1,6.4,10.480949,Fe ka
2,4,5.267717,Ca kb
3,7.04,4.266787,Fe kb
4,1.74,4.020145,Si
5,4.58,2.573184,Ti ka
6,5.86,2.320523,Mn
7,12.8,2.051473,sum Fe
8,1.46,1.852539,Al
9,5.22,1.845806,Cr


In [5]:
shap_global_importance = pd.read_csv('shap_soil.csv', sep=';') # loading previously saved shap_unique_df
# vamos gerar uma nova coluna em shap_global_importance com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
energy_to_zone_shap = {}
for zone_name, start, end in spectral_cuts:
    for i in shap_global_importance['energy']:
        i_float = float(i)
        if start <= i_float <= end:
            energy_to_zone_shap[i] = zone_name
shap_global_importance['Zone'] = shap_global_importance['energy'].map(energy_to_zone_shap)

# agora vamos filtrar shap_global_importance para manter apenas as zonas espectrais únicas com maior SHAP score
shap_unique_df = shap_global_importance.sort_values(by='Mean_Abs_SHAP', ascending=False).reset_index(drop=True)
shap_unique_df = shap_unique_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
shap_unique_df

,energy,Mean_Abs_SHAP,Zone
0,3.70,0.049862,Ca ka
1,6.44,0.012383,Fe ka
2,4.52,0.005719,Ti ka
3,1.74,0.002200,Si
4,5.92,0.001246,Mn
5,5.60,0.000197,Cr
6,10.24,0.000180,background2
7,4.00,0.000147,Ca kb
8,7.12,0.000120,Fe kb
9,4.96,0.000013,Ti kb


## Funções de Agregação por PCA

Aqui implementamos as funções para agregar zonas espectrais usando PCA com 1 componente principal.

In [6]:
import explaining as exp

# Extração das zonas espectrais
spectral_zones_class = exp.extract_spectral_zones(Xcalclass_prep, spectral_cuts)
zone_scores_df, pca_info_dict = exp.aggregate_spectral_zones_pca(spectral_zones_class) # apca aggregation
print(f"\nScores DataFrame shape: {zone_scores_df.shape}")

Zone 'Al': VE = 79.34%, dim = 15 variables
Zone 'Si': VE = 91.83%, dim = 12 variables
Zone 'P': VE = 62.02%, dim = 17 variables
Zone 'S': VE = 36.84%, dim = 18 variables
Zone 'Rh L + Ar': VE = 42.17%, dim = 33 variables
Zone 'K': VE = 83.48%, dim = 16 variables
Zone 'Ca ka': VE = 99.02%, dim = 16 variables
Zone 'Ca kb': VE = 74.05%, dim = 27 variables
Zone 'Ti ka': VE = 89.48%, dim = 19 variables
Zone 'Ti kb': VE = 82.21%, dim = 19 variables
Zone 'Cr': VE = 53.41%, dim = 33 variables
Zone 'Mn': VE = 91.12%, dim = 18 variables
Zone 'Fe ka': VE = 56.98%, dim = 34 variables
Zone 'Fe kb': VE = 55.98%, dim = 26 variables
Zone 'background1': VE = 32.70%, dim = 31 variables
Zone 'Cu': VE = 53.86%, dim = 15 variables
Zone 'background2': VE = 26.43%, dim = 125 variables
Zone 'Fe ka + Ti ka': VE = 34.74%, dim = 23 variables
Zone 'background3': VE = 18.96%, dim = 71 variables
Zone 'sum Fe': VE = 55.82%, dim = 28 variables

Scores DataFrame shape: (148, 20)


In [7]:
y_pred = svm_model[4]['SVC'].values # using the continuous predictions from SVM, extracting as 1D array
predicates_quantiles = exp.predicates_by_quantiles(zone_scores_df, [0.2, 0.4, 0.6, 0.8])
predicate_info_dict = exp.create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_scores_df,
    y_predicted_numeric=y_pred
)

In [8]:
import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_cov = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE

y_pred = pd.Series(svm_model[4]['SVC'].values) # using the continuous predictions from SVM, extracting as 1D array

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_scores_df,
        y_predicted_numeric=y_pred,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist
    
    # Calcular MI
    cov_results_dict_seed = exp.calculate_predicate_metrics(
        bags_result=bags_result_seed,
        metric='covariance', # covariance ou mutual_information
        threshold=0.01, # threshold para cortar predicados irrelevantes
        n_neighbors=5
    )
    
    # Salvar no dicionário principal
    all_results_cov[seed] = {
        'bags_result': bags_result_seed,
        'cov_results_dict': cov_results_dict_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    
    # Construir grafo para esta semente (sem predicates_df - extraído da regra)
    DG = exp.build_predicate_graph(
        bags_result=all_results_cov[seed]['bags_result'],
        predicate_ranking_dict=all_results_cov[seed]['cov_results_dict'],
        metric_column='Covariance',
        random_state=seed,
        show_details=True,
        var_exp=True,
        pca_info_dict=pca_info_dict
    )
    # Armazenar grafo
    graphs_by_seed[seed] = DG

# Calcular LRC usando a função pronta do explaining.py
lrc_cov_by_seed = {}
for seed in random_seeds:
    DG = graphs_by_seed[seed]
    lrc_cov_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_cov_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_cov_by_seed[seed] = lrc_cov_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_cov_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_cov_df_seed = lrc_cov_by_seed[seed].rename(columns={'Node': f'Predicate_Cov_Seed_{seed}'})
    lrc_cov_all_seeds_df = pd.concat([lrc_cov_all_seeds_df, lrc_cov_df_seed[[f'Predicate_Cov_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_cov_unique_by_seed = {}
for seed, lrc_df in lrc_cov_by_seed.items():
    lrc_cov_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_cov_unique_df = lrc_cov_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_cov_unique_by_seed[seed] = lrc_cov_unique_df

lrc_cov_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Samples: Yes | Predicates: No (All) | Valid: 123 | Discarded: 37
Bag_2 | Samples: Yes | Predicates: No (All) | Valid: 120 | Discarded: 40
Bag_3 | Samples: Yes | Predicates: No (All) | Valid: 120 | Discarded: 40
Bag_4 | Samples: Yes | Predicates: No (All) | Valid: 122 | Discarded: 38
Bag_5 | Samples: Yes | Predicates: No (All) | Valid: 120 | Discarded: 40
Bag_6 | Samples: Yes | Predicates: No (All) | Valid: 121 | Discarded: 39
Bag_7 | Samples: Yes | Predicates: No (All) | Valid: 120 | Discarded: 40
Bag_8 | Samples: Yes | Predicates: No (All) | Valid: 120 | Discarded: 40
Bag_9 | Samples: Yes | Predicates: No (All) | Valid: 120 | Discarded: 40
Bag_10 | Samples: Yes | Predicates: No (All) | Valid: 121 | Discarded: 39
Computing Covariance for Predicates
Metric: covariance
Threshold: 0.01

Processando semente: 1

Bag_1 | Samples: Yes | Predicates: No (All) | Valid: 121 | Discarded: 39
Bag_2 | Samples: Yes | Predicates: No (All) | Valid: 120 | Discarded: 40
Ba

,Predicate_Cov_Seed_0,Predicate_Cov_Seed_1,Predicate_Cov_Seed_2,Predicate_Cov_Seed_3
0,Ca ka > -1.73,Ca ka > -1.73,Ca ka > -1.73,Ca ka > -1.73
1,Ca ka > -1.02,Ca ka > -1.02,Ca ka <= 1.21,Ca ka > -1.02
2,Ca ka <= 1.21,Fe ka > 0.86,Ca ka > -1.02,Ca ka <= 1.21
3,Fe ka > 0.86,Ti ka > 0.93,Ti ka > 0.93,Ti ka > 0.93
4,Fe ka > -0.43,Ca ka <= 1.21,Fe ka > 0.86,Fe ka > -2.66
...,...,...,...,...
102,NaN,P <= 0.01,Rh L + Ar <= 0.09,NaN
103,NaN,Class_A,Cu <= 0.06,NaN
104,NaN,Class_B,background3 > -0.17,NaN
105,NaN,NaN,Class_A,NaN


In [9]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_cov_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df_cov = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df_cov = lrc_summed_df_cov.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_cov = lrc_summed_df_cov.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
#lrc_summed_unique_df_cov = lrc_summed_unique_df_cov.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_cov

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Ca ka > -1.73,14.158535,Ca ka,-1.73,>
1,Fe ka > 0.86,4.127752,Fe ka,0.86,>
2,Ti ka > 0.93,3.996832,Ti ka,0.93,>
3,Mn > -0.65,2.389461,Mn,-0.65,>
4,Si > -0.66,2.147667,Si,-0.66,>
5,Ca kb > -0.37,1.816538,Ca kb,-0.37,>
6,Fe kb > 0.40,1.502563,Fe kb,0.40,>
7,K > -0.38,1.373818,K,-0.38,>
8,Ti kb > -0.16,1.169004,Ti kb,-0.16,>
9,Al <= 0.37,1.145165,Al,0.37,<=


In [10]:
# Criar zone_sums_df para dados NÃO pré-processados (Xcalclass)
spectral_zones_original = exp.extract_spectral_zones(Xcalclass, spectral_cuts)
zones_original = exp.aggregate_spectral_zones_pca(spectral_zones_original) # apca aggregation

# Aplicar o mapeamento
lrc_summed_df_cov_natural = exp.map_thresholds_to_natural(
    lrc_df=lrc_summed_df_cov,
    zone_sums_preprocessed=zone_scores_df,
    zone_sums_natural=zones_original[0]
)

lrc_summed_df_cov_natural

Zone 'Al': VE = 89.82%, dim = 15 variables
Zone 'Si': VE = 95.22%, dim = 12 variables
Zone 'P': VE = 61.00%, dim = 17 variables
Zone 'S': VE = 42.01%, dim = 18 variables
Zone 'Rh L + Ar': VE = 51.54%, dim = 33 variables
Zone 'K': VE = 88.55%, dim = 16 variables
Zone 'Ca ka': VE = 99.40%, dim = 16 variables
Zone 'Ca kb': VE = 72.71%, dim = 27 variables
Zone 'Ti ka': VE = 93.59%, dim = 19 variables
Zone 'Ti kb': VE = 88.32%, dim = 19 variables
Zone 'Cr': VE = 56.23%, dim = 33 variables
Zone 'Mn': VE = 93.26%, dim = 18 variables
Zone 'Fe ka': VE = 58.12%, dim = 34 variables
Zone 'Fe kb': VE = 55.52%, dim = 26 variables
Zone 'background1': VE = 34.82%, dim = 31 variables
Zone 'Cu': VE = 55.93%, dim = 15 variables
Zone 'background2': VE = 26.61%, dim = 125 variables
Zone 'Fe ka + Ti ka': VE = 39.81%, dim = 23 variables
Zone 'background3': VE = 19.45%, dim = 71 variables
Zone 'sum Fe': VE = 69.07%, dim = 28 variables


,Node,Local_Reaching_Centrality,Zone,Threshold,Operator,Threshold_Natural,Reference_Sample_Index,Approximation_Error,Node_Natural
0,Ca ka > -1.73,14.158535,Ca ka,-1.73,>,-2.806273,140.0,0.001954,Ca ka > -2.806273
1,Ca ka > -1.02,6.133361,Ca ka,-1.02,>,-1.635267,143.0,0.000757,Ca ka > -1.635267
2,Ca ka <= 1.21,4.791394,Ca ka,1.21,<=,2.034067,2.0,0.019671,Ca ka <= 2.034067
3,Fe ka > 0.86,4.127752,Fe ka,0.86,>,55.595005,114.0,0.001174,Fe ka > 55.595005
4,Ti ka > 0.93,3.996832,Ti ka,0.93,>,6.770758,113.0,0.010621,Ti ka > 6.770758
...,...,...,...,...,...,...,...,...,...
109,Fe ka + Ti ka <= 0.06,0.002204,Fe ka + Ti ka,0.06,<=,0.049667,10.0,0.000571,Fe ka + Ti ka <= 0.049667
110,S <= -0.06,0.001321,S,-0.06,<=,-0.052921,96.0,0.001041,S <= -0.052921
111,P <= 0.01,0.000947,P,0.01,<=,-0.008542,69.0,0.002267,P <= -0.008542
112,Class_B,0.000000,None,None,None,NaN,NaN,NaN,None


## Reconstrução de Thresholds Multivariados

Agora vamos pegar os thresholds dos predicados mais importantes e reconstruí-los como espectros completos.

In [11]:
import plotly.graph_objects as go

n = 0
zone_name = lrc_summed_df_cov_natural.iloc[n]['Zone']
threshold_score = float(lrc_summed_df_cov_natural.iloc[n]['Threshold_Natural'])  # Corrigido: usa n em vez de 7

# Usar pca_info_dict dos dados ORIGINAIS (zones_original[1])
pca_info_dict_original = zones_original[1]

# Reconstrução: τ = mean + threshold * loadings (no espaço original)
threshold_spectrum = exp.reconstruct_threshold_to_spectrum(
    threshold_value=threshold_score,
    zone_name=zone_name,
    pca_info_dict=pca_info_dict_original
)
print(f"\nEspectro de threshold reconstruído para zona '{zone_name}':")
print(f"  - Dimensão: {len(threshold_spectrum)} variáveis espectrais")
print(f"  - Range de energias: {threshold_spectrum.index[0]} - {threshold_spectrum.index[-1]}")
print(f"  - Variância explicada pela PC1: {pca_info_dict_original[zone_name]['variance_explained']:.2%}")

# Usar dados ORIGINAIS para plotar (spectral_zones_original)
zone_df = spectral_zones_original[zone_name]

# Plotar zona com threshold (inline)
fig = go.Figure()
x_values = pd.to_numeric(zone_df.columns, errors='coerce')

# Plotar espectros das amostras coloridas por classe
colors = {'A': 'gold', 'B': 'blue'}
for idx, row in zone_df.iterrows():
    class_label = ycalclass.iloc[idx] if idx < len(ycalclass) else 'Unknown'
    fig.add_trace(go.Scatter(
        x=x_values,
        y=row.values,
        mode='lines',
        line=dict(color=colors.get(class_label, 'rgba(128,128,128,0.3)'), width=0.5),
        name=f'Class {class_label}',
        showlegend=False,
        hoverinfo='skip'
    ))

# Plotar espectro de threshold (destaque)
fig.add_trace(go.Scatter(
    x=x_values,
    y=threshold_spectrum.values,
    mode='lines',
    line=dict(color='red', width=4, dash='dash'),
    name=f'Threshold Spectrum ({threshold_spectrum.name})'
))

# Layout do gráfico
fig.update_layout(
    title=f"Zona '{zone_name}' com Threshold Multivariado (Predicado: {lrc_summed_df_cov_natural.iloc[n]['Node_Natural']})",
    xaxis_title='Energia / Comprimento de Onda',
    yaxis_title='Intensidade',
    template='plotly_white',
    showlegend=True,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()


Espectro de threshold reconstruído para zona 'Ca ka':
  - Dimensão: 16 variáveis espectrais
  - Range de energias: 3.54 - 3.84
  - Variância explicada pela PC1: 99.40%


In [12]:
import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_pert = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_scores_df,
        y_predicted_numeric=y_pred,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist

    pert_results_seed = exp.calculate_predicate_perturbation(
        estimator=svm_model[3],
        Xcalclass_prep=Xcalclass_prep,
        folds_struct=bags_result_seed,
        predicates_df=predicates_quantiles[0],
        spectral_cuts=spectral_cuts,
        #perturbation_value=0,
        perturbation_mode='median', # valores entre 'mean' ou 'min'
        stats_source='full', # full indica usar todas as amostras para calcular estatísticas enquanto que 'fold' usa apenas as amostras do fold atual
        #metric='mean_abs_diff',   # Média com sinal (pode ser negativo)
        aim='classification',
        metric='probability_shift', 
        verbose=True
    )

    # Salvar no dicionário principal
    all_results_pert[seed] = {
        'bags_result': bags_result_seed,
        'pert_results_dict': pert_results_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_pert_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    
    # Construir grafo para esta semente (sem predicates_df - extraído da regra)
    DG = exp.build_predicate_graph(
        bags_result=all_results_pert[seed]['bags_result'],
        predicate_ranking_dict=all_results_pert[seed]['pert_results_dict'],
        metric_column='Perturbation',
        random_state=seed,
        show_details=True,
        var_exp=True,
        pca_info_dict=pca_info_dict
    )

    # Armazenar grafo
    graphs_pert_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_pert_by_seed = {}
for seed in random_seeds:
    DG = graphs_pert_by_seed[seed]
    lrc_pert_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_pert_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_pert_by_seed[seed] = lrc_pert_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_pert_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_pert_df_seed = lrc_pert_by_seed[seed].rename(columns={'Node': f'Predicate_pert_Seed_{seed}'})
    lrc_pert_all_seeds_df = pd.concat([lrc_pert_all_seeds_df, lrc_pert_df_seed[[f'Predicate_pert_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_pert_unique_by_seed = {}
for seed, lrc_df in lrc_pert_by_seed.items():
    lrc_pert_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_pert_unique_df = lrc_pert_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_pert_unique_by_seed[seed] = lrc_pert_unique_df

lrc_pert_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Samples: Yes | Predicates: No (All) | Valid: 123 | Discarded: 37
Bag_2 | Samples: Yes | Predicates: No (All) | Valid: 120 | Discarded: 40
Bag_3 | Samples: Yes | Predicates: No (All) | Valid: 120 | Discarded: 40
Bag_4 | Samples: Yes | Predicates: No (All) | Valid: 122 | Discarded: 38
Bag_5 | Samples: Yes | Predicates: No (All) | Valid: 120 | Discarded: 40
Bag_6 | Samples: Yes | Predicates: No (All) | Valid: 121 | Discarded: 39
Bag_7 | Samples: Yes | Predicates: No (All) | Valid: 120 | Discarded: 40
Bag_8 | Samples: Yes | Predicates: No (All) | Valid: 120 | Discarded: 40
Bag_9 | Samples: Yes | Predicates: No (All) | Valid: 120 | Discarded: 40
Bag_10 | Samples: Yes | Predicates: No (All) | Valid: 121 | Discarded: 39
PERTURBATION IMPORTANCE FOR PREDICATES
Task type (aim): classification
Perturbation mode: median
Statistics source: full
Metric: probability_shift
Total folds: 10


[Bag_1] Processing 123 predicates...
  Predicate: Al > -0.33 (n=93)
    Zone: 1

,Predicate_pert_Seed_0,Predicate_pert_Seed_1,Predicate_pert_Seed_2,Predicate_pert_Seed_3
0,Ca ka > -0.36,Ca ka > -0.36,Ca ka > -0.36,Ca ka > -0.36
1,Ca ka > -1.02,Ca ka > -1.02,Ca ka > -1.02,Ca ka > -1.02
2,Ca ka > -1.73,Ca ka > -1.73,Ca ka > -1.73,Ca ka > -1.73
3,Ca ka <= -1.02,Ca ka <= -1.02,Ca ka <= -1.02,Ca ka <= -1.02
4,Ca ka <= 1.21,Ca ka <= 1.21,Ca ka <= 1.21,Ca ka <= 1.21
...,...,...,...,...
124,Fe ka + Ti ka > -0.04,NaN,Fe ka + Ti ka > -0.15,Fe ka + Ti ka <= 0.16
125,Fe ka + Ti ka <= 0.16,NaN,Class_A,Class_A
126,Fe ka + Ti ka > -0.15,NaN,Class_B,Class_B
127,Class_A,NaN,NaN,NaN


In [13]:
all_results_pert[0]['pert_results_dict']['Bag_1']

,Predicate,Perturbation
0,Ca ka > -0.36,0.289381
1,Ca ka > -1.02,0.214587
2,Ca ka > -1.73,0.187820
3,Ca ka <= -1.02,0.162082
4,Ca ka <= 1.21,0.151767
...,...,...
118,Fe ka + Ti ka <= 0.06,0.000654
119,Fe ka + Ti ka > 0.06,0.000628
120,Fe ka + Ti ka <= 0.16,0.000620
121,Fe ka + Ti ka > -0.04,0.000572


In [14]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_pert_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df_pert = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df_pert = lrc_summed_df_pert.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_pert = lrc_summed_df_pert.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
lrc_summed_unique_df_pert = lrc_summed_unique_df_pert.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_pert

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Ca ka > -0.36,20.512111,Ca ka,-0.36,>
1,Fe ka > 0.86,2.848046,Fe ka,0.86,>
2,Ti ka > 0.93,2.290338,Ti ka,0.93,>
3,Mn > 0.64,1.130747,Mn,0.64,>
4,Si > 0.13,1.087170,Si,0.13,>
5,Fe kb <= -0.07,0.502463,Fe kb,-0.07,<=
6,Ti kb <= -0.63,0.398855,Ti kb,-0.63,<=
7,Ca kb <= -0.23,0.206218,Ca kb,-0.23,<=
8,background2 <= -0.29,0.193309,background2,-0.29,<=
9,S <= -0.13,0.096174,S,-0.13,<=


In [15]:
# Aplicar o mapeamento
lrc_summed_df_pert_natural = exp.map_thresholds_to_natural(
    lrc_df=lrc_summed_df_pert,
    zone_sums_preprocessed=zone_scores_df,
    zone_sums_natural=zones_original[0]
)

lrc_summed_df_pert_natural

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator,Threshold_Natural,Reference_Sample_Index,Approximation_Error,Node_Natural
0,Ca ka > -0.36,20.512111,Ca ka,-0.36,>,-0.641296,49.0,0.004141,Ca ka > -0.641296
1,Ca ka > -1.02,15.992136,Ca ka,-1.02,>,-1.635267,143.0,0.000757,Ca ka > -1.635267
2,Ca ka > -1.73,12.989248,Ca ka,-1.73,>,-2.806273,140.0,0.001954,Ca ka > -2.806273
3,Ca ka <= -1.02,10.449417,Ca ka,-1.02,<=,-1.635267,143.0,0.000757,Ca ka <= -1.635267
4,Ca ka <= 1.21,8.036128,Ca ka,1.21,<=,2.034067,2.0,0.019671,Ca ka <= 2.034067
...,...,...,...,...,...,...,...,...,...
131,Fe ka + Ti ka > -0.04,0.001336,Fe ka + Ti ka,-0.04,>,-0.040286,48.0,0.000022,Fe ka + Ti ka > -0.040286
132,Fe ka + Ti ka > -0.15,0.000736,Fe ka + Ti ka,-0.15,>,-0.137035,129.0,0.007042,Fe ka + Ti ka > -0.137035
133,Fe ka + Ti ka <= 0.16,0.000682,Fe ka + Ti ka,0.16,<=,0.163402,23.0,0.000909,Fe ka + Ti ka <= 0.163402
134,Class_B,0.000000,None,None,None,NaN,NaN,NaN,None


In [16]:
import plotly.graph_objects as go

n = 0
zone_name = lrc_summed_df_pert_natural.iloc[n]['Zone'] # o n indica a linha do dataframe lrc_summed_df_pert_natural que queremos acessar para pegar o nome da zona espectral (coluna 'Zone')
threshold_score = float(lrc_summed_df_pert_natural.iloc[n]['Threshold_Natural']) # o iloc acessa a linha n e a coluna 'Threshold_Natural' para pegar o valor do threshold já mapeado para o espaço natural (não pré-processado)

# Usar pca_info_dict dos dados ORIGINAIS (zones_original[1])
pca_info_dict_original = zones_original[1]

# Reconstrução: τ = mean + threshold * loadings (no espaço original)
threshold_spectrum = exp.reconstruct_threshold_to_spectrum(
    threshold_value=threshold_score,
    zone_name=zone_name,
    pca_info_dict=pca_info_dict_original
)
print(f"\nEspectro de threshold reconstruído para zona '{zone_name}':")
print(f"  - Dimensão: {len(threshold_spectrum)} variáveis espectrais")
#print(f"  - Range de energias: {threshold_spectrum.index[n]} - {threshold_spectrum.index[-1]}")
print(f"  - Variância explicada pela PC1: {pca_info_dict_original[zone_name]['variance_explained']:.2%}")

# Usar dados ORIGINAIS para plotar (spectral_zones_original)
zone_df = spectral_zones_original[zone_name]

# Plotar zona com threshold (inline)
fig = go.Figure()
x_values = pd.to_numeric(zone_df.columns, errors='coerce')

# Plotar espectros das amostras coloridas por classe
colors = {'A': 'gold', 'B': 'blue'}
for idx, row in zone_df.iterrows():
    class_label = ycalclass.iloc[idx] if idx < len(ycalclass) else 'Unknown'
    fig.add_trace(go.Scatter(
        x=x_values,
        y=row.values,
        mode='lines',
        line=dict(color=colors.get(class_label, 'rgba(128,128,128,0.3)'), width=0.5),
        name=f'Class {class_label}',
        showlegend=False,
        hoverinfo='skip'
    ))

# Plotar espectro de threshold (destaque)
fig.add_trace(go.Scatter(
    x=x_values,
    y=threshold_spectrum.values,
    mode='lines',
    line=dict(color='red', width=4, dash='dash'),
    name=f'Threshold Spectrum ({threshold_spectrum.name})'
))

# Layout do gráfico
fig.update_layout(
    title=f"Zona '{zone_name}' com Threshold Multivariado (Predicado: {lrc_summed_df_pert_natural.iloc[n]['Node_Natural']})",
    xaxis_title='Energia / Comprimento de Onda',
    yaxis_title='Intensidade',
    template='plotly_white',
    showlegend=True,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()


Espectro de threshold reconstruído para zona 'Ca ka':
  - Dimensão: 16 variáveis espectrais
  - Variância explicada pela PC1: 99.40%


In [17]:
# Permutation importance baseado em mudança nas probabilidades previstas (predict_proba)
# Medimos a média da diferença absoluta entre as probabilidades originais e as probabilidades
# obtidas após permutar cada variável. Isso fornece uma métrica contínua de importância.
n_repeats = 10
rng = np.random.RandomState(42)
# Probabilidades base (classe 'B' mapeada para 1)
baseline_proba = svm_model[3].predict_proba(Xcalclass_prep)[:, 1]
importance_list = []
X_arr = Xcalclass_prep.copy()
for col in Xcalclass_prep.columns:
    diffs = []
    for _ in range(n_repeats):
        X_perm = X_arr.copy()
        X_perm[col] = rng.permutation(X_perm[col].values)
        perm_proba = svm_model[3].predict_proba(X_perm)[:, 1]
        diffs.append(np.mean(np.abs(baseline_proba - perm_proba)))
    importance_list.append(np.mean(diffs))

permutation_df = pd.DataFrame({
    'energy': Xcalclass_prep.columns,
    'Permutation_importance_proba': importance_list
})
permutation_df.sort_values(by='Permutation_importance_proba', ascending=False, inplace=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in permutation_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
permutation_df['Zone'] = permutation_df['energy'].map(energy_to_zone_vip)
permutation_unique_df = permutation_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
permutation_unique_df = permutation_unique_df.sort_values(by='Permutation_importance_proba', ascending=False)
permutation_unique_df

,energy,Permutation_importance_proba,Zone
0,3.7,0.043557,Ca ka
1,6.48,0.017673,Fe ka
2,4.52,0.010735,Ti ka
3,1.74,0.005132,Si
4,5.9,0.003679,Mn
5,7.14,0.003100,Fe kb
6,4.02,0.002621,Ca kb
7,4.96,0.001283,Ti kb
8,2.02,0.001207,P
9,1.54,0.001110,Al


In [18]:
import numpy as np

max_len = max(
    len(pvector_unique_df['Zone']),
    len(shap_unique_df['Zone']),
    len(permutation_unique_df['Zone']),
    len(lrc_summed_unique_df_pert['Zone']),
    len(lrc_summed_unique_df_cov['Zone'])
)

def pad_list(lst, length):
    return list(lst) + [None] * (length - len(lst))

features_importance = pd.DataFrame({
    'SVM_pvector': pad_list(pvector_unique_df['Zone'], max_len),
    'Shap': pad_list(shap_unique_df['Zone'], max_len),
    'Permutation' : pad_list(permutation_unique_df['Zone'], max_len),
    'LRC_perturbation' : pad_list(lrc_summed_unique_df_pert['Zone'], max_len),
    'LRC_covariance' : pad_list(lrc_summed_unique_df_cov['Zone'], max_len),
})

features_importance.to_csv('feature_importance.csv', index=False, sep=';')
features_importance

,SVM_pvector,Shap,Permutation,LRC_perturbation,LRC_covariance
0,Ca ka,Ca ka,Ca ka,Ca ka,Ca ka
1,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka
2,Ca kb,Ti ka,Ti ka,Ti ka,Ti ka
3,Fe kb,Si,Si,Mn,Mn
4,Si,Mn,Mn,Si,Si
5,Ti ka,Cr,Fe kb,Fe kb,Ca kb
6,Mn,background2,Ca kb,Ti kb,Fe kb
7,sum Fe,Ca kb,Ti kb,Ca kb,K
8,Al,Fe kb,P,background2,Ti kb
9,Cr,Ti kb,Al,S,Al


In [19]:
import rbo
rbo_comparison = pd.DataFrame(columns=['Method_1', 'Method_2', 'RBO_Score'])
methods = features_importance.columns.tolist()
for i in range(len(methods)):
    for j in range(i + 1, len(methods)):
        method_1 = methods[i]
        method_2 = methods[j]
        # Remove None values from the lists
        list_1 = [x for x in features_importance[method_1].tolist() if x is not None]
        list_2 = [x for x in features_importance[method_2].tolist() if x is not None]
        # Truncate both lists to the same length (minimum of both)
        min_len = min(len(list_1), len(list_2))
        list_1_trunc = list_1[:min_len]
        list_2_trunc = list_2[:min_len]
        rbo_score = rbo.RankingSimilarity(list_1_trunc, list_2_trunc).rbo(p=0.7, k=10)
        rbo_comparison = pd.concat([rbo_comparison, pd.DataFrame({
            'Method_1': [method_1],
            'Method_2': [method_2],
            'RBO_Score': [rbo_score]
        })], ignore_index=True)
rbo_comparison.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_comparison.to_csv('rbo_rank.csv', index=False, sep=';')
rbo_comparison

C:\Users\Usuario\AppData\Local\Temp\ipykernel_13528\3097109587.py:16: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



,Method_1,Method_2,RBO_Score
9,LRC_perturbation,LRC_covariance,0.950876
4,Shap,Permutation,0.940824
7,Permutation,LRC_perturbation,0.936643
8,Permutation,LRC_covariance,0.931403
5,Shap,LRC_perturbation,0.918231
6,Shap,LRC_covariance,0.915099
1,SVM_pvector,Permutation,0.824734
3,SVM_pvector,LRC_covariance,0.824734
2,SVM_pvector,LRC_perturbation,0.818482
0,SVM_pvector,Shap,0.803158


In [20]:
lrc_summed_df_cov_natural.to_csv('lrc_cov_natural.csv', index=False, sep=';')
lrc_summed_df_pert_natural.to_csv('lrc_pert_natural.csv', index=False, sep=';')